# Lab 3.1 — API/SDK Call Differences (Runtime vs Mantle)

**Duration:** ~8 min · **Level:** L300 · **Lab 3 of 4 — part 1/4**

You're here because you have production code that calls **Amazon Bedrock
Runtime** (`bedrock-runtime.{region}.amazonaws.com`) — most likely via the
`Converse` or `InvokeModel` operations — and you need to migrate those call
sites to **Mantle** (`bedrock-mantle.{region}.api.aws`).

This notebook runs the *same prompt* through:

1. `bedrock-runtime` `Converse` — the source of truth you're migrating from.
2. Mantle **Chat Completions** — the most common target for
   `gpt-oss`, Mistral, GLM, Qwen, MiniMax, DeepSeek, …
3. Mantle **Anthropic Messages** — the target for Claude Opus 4.7 /
   Haiku 4.5 / Mythos.

After you see the three shapes side-by-side, you'll know exactly which
response fields change and where.

> **Prerequisite:** your account must have **both** runtime model access
> (`mistral.mistral-large-2407-v1:0` or a similar Converse-capable model —
> we fall back gracefully if absent) *and* Mantle access to
> `openai.gpt-oss-120b` + `anthropic.claude-opus-4-7`.


In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

import boto3
from botocore.exceptions import ClientError
from src.common.mantle import (
    openai_client, anthropic_client,
    GPT_OSS_120B, CLAUDE_OPUS_47,
)

runtime = boto3.client("bedrock-runtime", region_name=os.environ["AWS_REGION"])
mantle_oai = openai_client()
mantle_anth = anthropic_client()

PROMPT = "In three words, what is AWS Nitro?"
print("ready")


## 1. Before — `bedrock-runtime` Converse

Converse is the unified chat API on the legacy endpoint. Key shapes:

- `messages=[{role, content:[{text: "..."}]}]` (a list of **content blocks**,
  not a plain string).
- `system=[{text: "..."}]` as a sibling field.
- `inferenceConfig={"maxTokens": ..., "temperature": ..., "topP": ...}`.
- Response at `output.message.content[0].text`.
- Token counts at `usage.inputTokens` / `usage.outputTokens`.
- `stopReason ∈ {end_turn, tool_use, max_tokens, stop_sequence, guardrail_intervened, content_filtered}`.


In [ ]:
# Use a stable Claude on runtime as the source baseline. If your account
# doesn't have this model, substitute any Converse-capable model ID.
RUNTIME_MODEL_CANDIDATES = [
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-7",                 # some accounts route Opus 4.7 here too
    "mistral.mistral-large-2407-v1:0",
    "anthropic.claude-3-5-haiku-20241022-v1:0",  # broadly-accessible fallback
]

runtime_resp = None
chosen = None
for mid in RUNTIME_MODEL_CANDIDATES:
    try:
        runtime_resp = runtime.converse(
            modelId=mid,
            system=[{"text": "Answer in exactly three words."}],
            messages=[{"role": "user", "content": [{"text": PROMPT}]}],
            inferenceConfig={"maxTokens": 40, "temperature": 0.2, "topP": 0.9},
        )
        chosen = mid
        break
    except ClientError as e:
        print(f"  {mid}: {e.response['Error']['Code']}")
        continue

if runtime_resp is None:
    print("⚠️  No Converse-capable model available — skipping runtime baseline.")
else:
    print(f"used runtime model: {chosen}")
    text = runtime_resp["output"]["message"]["content"][0]["text"]
    u = runtime_resp["usage"]
    print(f"text         : {text!r}")
    print(f"inputTokens  : {u['inputTokens']}")
    print(f"outputTokens : {u['outputTokens']}")
    print(f"stopReason   : {runtime_resp['stopReason']}")


## 2. After — Mantle **Chat Completions**

Shape changes:

- `messages=[{role, content: str}]` — plain strings, not content blocks.
- No `system` field; the system message is just `role: "system"` in
  `messages`.
- Top-level `max_tokens`, `temperature`, `top_p` (not nested).
- Response at `choices[0].message.content` — a plain string.
- Tokens at `usage.prompt_tokens` / `usage.completion_tokens`.
- `finish_reason ∈ {stop, tool_calls, length, content_filter}`. (The
  legacy `function_call` value is only emitted by the deprecated
  single-function API, not the current `tools`-style calls.)


In [ ]:
cc = mantle_oai.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[
        {"role": "system", "content": "Answer in exactly three words."},
        {"role": "user",   "content": PROMPT},
    ],
    max_tokens=40,
    temperature=0.2,
    top_p=0.9,
)
text = cc.choices[0].message.content
print(f"text              : {text!r}")
print(f"prompt_tokens     : {cc.usage.prompt_tokens}")
print(f"completion_tokens : {cc.usage.completion_tokens}")
print(f"finish_reason     : {cc.choices[0].finish_reason}")


## 3. After — Mantle **Anthropic Messages**

Shape changes (vs Converse):

- `system="..."` is a *string*, not a list of content blocks.
- `messages=[{role, content: str_or_blocks}]` — strings are fine for plain
  text; blocks (`[{type:"text", ...}]`) are required for images / docs.
- Top-level `max_tokens` (not nested).
- Response at `content[0].text`.
- Tokens at `usage.input_tokens` / `usage.output_tokens`.
- `stop_reason ∈ {end_turn, tool_use, max_tokens, stop_sequence, refusal}`.


In [ ]:
msg = mantle_anth.messages.create(
    model=CLAUDE_OPUS_47,
    max_tokens=40,
    system="Answer in exactly three words.",
    messages=[{"role": "user", "content": PROMPT}],
    # Opus 4.7: NO temperature / top_p / top_k.
)
print(f"text         : {msg.content[0].text!r}")
print(f"input_tokens : {msg.usage.input_tokens}")
print(f"output_tokens: {msg.usage.output_tokens}")
print(f"stop_reason  : {msg.stop_reason}")


## 4. The "one table to rule them all"

| Concept | Converse (runtime) | Chat Completions (Mantle) | Anthropic Messages (Mantle) |
|---|---|---|---|
| Messages container | `messages: [{role, content:[{text}]}]` | `messages: [{role, content: "str"}]` | `messages: [{role, content: "str" or [...]}]` |
| System prompt | `system: [{text:"..."}]` | `role:"system"` message | `system: "..."` (sibling) |
| Max output tokens | `inferenceConfig.maxTokens` | `max_tokens` | `max_tokens` |
| Temperature | `inferenceConfig.temperature` | `temperature` | `temperature` |
| Top-p | `inferenceConfig.topP` | `top_p` | `top_p` |
| Provider-extra params | `additionalModelRequestFields` | `extra_body` | native fields (`thinking`) |
| Primary text | `output.message.content[0].text` | `choices[0].message.content` | `content[0].text` |
| Input tokens | `usage.inputTokens` | `usage.prompt_tokens` | `usage.input_tokens` |
| Output tokens | `usage.outputTokens` | `usage.completion_tokens` | `usage.output_tokens` |
| Stop reason | `stopReason` | `finish_reason` | `stop_reason` |

If you only memorise one thing from this lab: **every response field name
changes across the three surfaces.** Build a provider-abstraction layer
that normalises them or you'll debug this forever.


## 5. A minimal provider abstraction

A rough shape you can grow into production:


In [ ]:
from dataclasses import dataclass

@dataclass
class NormalizedReply:
    text: str
    input_tokens: int
    output_tokens: int
    stop_reason: str
    surface: str           # "converse" / "chat_completions" / "messages"

def from_converse(r) -> NormalizedReply:
    return NormalizedReply(
        text=r["output"]["message"]["content"][0]["text"],
        input_tokens=r["usage"]["inputTokens"],
        output_tokens=r["usage"]["outputTokens"],
        stop_reason=r["stopReason"],
        surface="converse",
    )

def from_chat_completions(r) -> NormalizedReply:
    return NormalizedReply(
        text=r.choices[0].message.content,
        input_tokens=r.usage.prompt_tokens,
        output_tokens=r.usage.completion_tokens,
        stop_reason=r.choices[0].finish_reason,
        surface="chat_completions",
    )

def from_messages(r) -> NormalizedReply:
    return NormalizedReply(
        text=next((b.text for b in r.content if b.type == "text"), ""),
        input_tokens=r.usage.input_tokens,
        output_tokens=r.usage.output_tokens,
        stop_reason=r.stop_reason,
        surface="messages",
    )

normalized = [
    from_chat_completions(cc),
    from_messages(msg),
]
if runtime_resp is not None:
    normalized.insert(0, from_converse(runtime_resp))

for n in normalized:
    print(f"[{n.surface:>18}] in={n.input_tokens} out={n.output_tokens} stop={n.stop_reason} text={n.text!r}")


## 6. What's next

The remaining Lab 3 notebooks go deep on each migration axis:

- `02_auth_security_migration.ipynb` — IAM policies, Bearer tokens, CloudTrail.
- `03_tools_and_caching_migration.ipynb` — Converse tool loop → Chat
  Completions tool loop, and caching diffs.
- `04_perf_eval.ipynb` — a runnable TTFT / tokens-per-second benchmark.
